# load_from_ddb

DynamoDB `gsretail-mlops-edu-hjsong`에서 conf와 data를 내려받아
로컬 `ddb/conf/`, `ddb/data/` 파일을 생성합니다.

기존 `modeling/modeling.ipynb` 실행 전 준비 단계로 사용합니다.

## 읽을 아이템
```
experiment_id = EXP#{user_id}#{project}#{experiment}
entity_type   = CONF            → env/meta/model YAML 복원
entity_type   = DATA#train      → train.csv 복원
entity_type   = DATA#validation → validation.csv 복원
entity_type   = DATA#test       → test.csv 복원
```

## 1. 라이브러리 및 연결 설정

In [ ]:
import io, yaml, base64, boto3
from pathlib import Path
from decimal import Decimal

BASE_DIR = Path.cwd()   # ddb/

TABLE_NAME = 'gsretail-mlops-edu-hjsong'
REGION     = 'us-east-1'
USER_ID    = 'hjsong'
PROJECT    = 'titanic-survival-prediction'
EXPERIMENT = 'tuned-hjsong-v1'

# experiment_id 키: EXP#{user_id}#{project}#{experiment}
EXP_PK = f'EXP#{USER_ID}#{PROJECT}#{EXPERIMENT}'

ddb   = boto3.resource('dynamodb', region_name=REGION)
table = ddb.Table(TABLE_NAME)

print(f'Table        : {TABLE_NAME}')
print(f'experiment_id: {EXP_PK}')

## 2. DynamoDB 타입 변환 헬퍼

DynamoDB에서 읽은 숫자는 `Decimal` 타입입니다.
YAML로 저장하려면 `float`/`int`로 변환해야 합니다.

In [ ]:
def from_ddb(obj):
    """Decimal → int/float 재귀 변환 (DynamoDB 로드 후 파이썬 표준 타입으로)"""
    if isinstance(obj, Decimal):
        f = float(obj)
        return int(f) if f == int(f) else f
    elif isinstance(obj, dict):
        return {k: from_ddb(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [from_ddb(v) for v in obj]
    return obj

print('from_ddb 헬퍼 정의 완료')

## 3. conf/ 생성 (CONF 아이템 → YAML 파일)

DynamoDB CONF 아이템에서 env_yml / meta_yml / model_yml 을 꺼내
로컬 YAML 파일로 저장합니다.

In [ ]:
CONF_DIR = BASE_DIR / 'conf'
CONF_DIR.mkdir(parents=True, exist_ok=True)

# CONF 아이템 조회
resp      = table.get_item(Key={'experiment_id': EXP_PK, 'entity_type': 'CONF'})
conf_item = from_ddb(resp['Item'])   # Decimal → float/int 변환

# 각 YAML 파일 복원
for name, data in [('env',  conf_item['env_yml']),
                   ('meta', conf_item['meta_yml']),
                   ('model',conf_item['model_yml'])]:
    path = CONF_DIR / f'{name}.yml'
    with open(path, 'w', encoding='utf-8') as f:
        yaml.dump(data, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
    print(f'  [저장] {path.name}  ({path.stat().st_size} bytes)')

print('conf/ 생성 완료')

## 4. data/ 생성 (DATA#split 아이템 → CSV 파일)

DynamoDB DATA 아이템에서 `csv_b64` 필드를 꺼내
`base64.b64decode()` → CSV 파일로 복원합니다.

In [ ]:
import pandas as pd

DATA_DIR = BASE_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

for split in ['train', 'validation', 'test']:
    # DATA#split 아이템 조회
    resp = table.get_item(Key={'experiment_id': EXP_PK, 'entity_type': f'DATA#{split}'})
    item = resp['Item']

    # base64 → bytes → DataFrame
    csv_bytes = base64.b64decode(item['csv_b64'])
    df        = pd.read_csv(io.BytesIO(csv_bytes))

    path = DATA_DIR / f'{split}.csv'
    df.to_csv(path, index=False)
    print(f'  [저장] {path.name}  {len(df)}rows  {path.stat().st_size/1024:.1f}KB')

print('data/ 생성 완료')

## 5. 검증

In [ ]:
print('=' * 60)
for p in [CONF_DIR/'env.yml', CONF_DIR/'meta.yml', CONF_DIR/'model.yml',
          DATA_DIR/'train.csv', DATA_DIR/'validation.csv', DATA_DIR/'test.csv']:
    ok = p.exists()
    size = f'{p.stat().st_size/1024:.1f}KB' if ok else '-'
    print(f'  {"OK" if ok else "MISSING"}: {p.name}  ({size})')
print('=' * 60)
print('다음 단계: modeling/modeling.ipynb 또는 ddb/modeling_ddb.ipynb')